# 02 — Prepare Sales (Parameterized)
## Goal
- standardize the sales data
- clean types
- create a silver table
- print useful run context

In [0]:
import yaml
import sys

from pyspark.sql import functions as F

sys.path.append("/Workspace/Repos/adb-tetiana@startsteps.org/beiersdorf-live-demo/src")

from transformations.sales_filters import (
    filter_report_date_range,
    filter_market,
    filter_brand
)

In [0]:
config_path = "/Workspace/Repos/adb-tetiana@startsteps.org/beiersdorf-live-demo/config/project_config.yml"

In [0]:
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

In [0]:
catalog_name = config["catalog"]
schema_name = config["schema"]

bronze_sales_table = f"{catalog_name}.{schema_name}.{config['tables']['bronze_sales']}"
silver_sales_table = f"{catalog_name}.{schema_name}.{config['tables']['silver_sales']}"

high_value_threshold = config["rules"]["high_value_threshold"]
allowed_markets = config["rules"]["allowed_markets"]
allowed_brands = config["rules"]["allowed_brands"]

print("Bronze sales table:", bronze_sales_table)
print("Silver sales table:", silver_sales_table)
print("High value threshold:", high_value_threshold)

In [0]:
dbutils.widgets.text(name="report_start_date", defaultValue="2026-04-25", label="Report Start Date")
dbutils.widgets.text(name="report_end_date", defaultValue="2026-04-30", label="Report End Date")

In [0]:
dbutils.widgets.dropdown(name= "market", defaultValue= "ALL", choices= ["ALL", "DE", "FR", "ES", "IT"], label="Market")

dbutils.widgets.dropdown("brand", "ALL", ["ALL", "NIVEA", "EUCERIN", "LABELLO", "HANSAPLAST"], "Brand")

In [0]:
report_start_date = dbutils.widgets.get("report_start_date")
report_end_date = dbutils.widgets.get("report_end_date")
market = dbutils.widgets.get("market")
brand = dbutils.widgets.get("brand")


print("Report Start date:", report_start_date)
print("Report end date:", report_end_date)
print("Market:", market)
print("Brand:", brand)

## Validate widget values

In [0]:
if market not in allowed_markets:
    raise ValueError(f"Unsupported market: {market}")

if brand not in allowed_brands:
    raise ValueError(f"Unsupported brand: {brand}")

## Read bronze sales data

In [0]:
bronze_sales_df = spark.table(bronze_sales_table)

print("Bronze row count before notebook filtering:", bronze_sales_df.count())
display(bronze_sales_df)

## Apply notebook filtering again for visibility and reuse
This keeps the notebook self-contained and makes the selected run scope explicit.

In [0]:
# filtered_sales_df = bronze_sales_df.filter(
#    (bronze_sales_df.report_date >= report_start_date) &
#    (bronze_sales_df.report_date <= report_end_date) )

# if market != "ALL":
#    filtered_sales_df = filtered_sales_df.filter(bronze_sales_df.market == market)

# if brand != "ALL":
#    filtered_sales_df = filtered_sales_df.filter(bronze_sales_df.brand == brand)

# filtered_count = filtered_sales_df.count()
# print("Rows after bronze filtering:", filtered_count)

# display(filtered_sales_df)
filtered_sales_df = (
    bronze_sales_df
    .transform(filter_report_date_range(report_start_date, report_end_date))
    .transform(filter_market(market))
    .transform(filter_brand(brand))
)

filtered_count = filtered_sales_df.count()
print("Rows after bronze filtering:", filtered_count)

display(filtered_sales_df)

## Clean and standardize the sales data

In [0]:
silver_sales_df = (
    filtered_sales_df
    .withColumn("sales_amount", F.col("sales_amount").cast("double"))
    .withColumn("units_sold", F.col("units_sold").cast("int"))
    .withColumn("inventory_level", F.col("inventory_level").cast("int"))
    .withColumn("report_day", F.to_date("report_date"))
    .withColumn("market", F.upper(F.trim(F.col("market"))))
    .withColumn("brand", F.upper(F.trim(F.col("brand"))))
    .withColumn("product_category", F.upper(F.trim(F.col("product_category"))))
    .filter(F.col("report_day").isNotNull())
    .filter(F.col("sales_amount").isNotNull())
    .filter(F.col("units_sold").isNotNull())
    .filter(F.col("inventory_level").isNotNull())
    .filter(F.col("sales_amount") >= 0)
    .withColumn(
        "is_high_value_market_row",
        F.when(F.col("sales_amount") >= high_value_threshold, 1).otherwise(0)
    )
)

silver_count = silver_sales_df.count()

print("Silver row count after cleaning:", silver_count)
display(silver_sales_df)


## Write the silver sales table

In [0]:
(
    silver_sales_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_sales_table)
)

In [0]:
print("Silver sales table written:", silver_sales_table)

In [0]:
display(spark.table(silver_sales_table))